# 注意力机制（下）自注意力与 Transformer Encoder

chap8 上的加性注意力是 query 来自分类层 / decoder、key&value 来自 encoder——典型的 "encoder-decoder attention"。

**自注意力** (self-attention) 让序列里的每个位置都拿自己的 query 去 attend 所有位置的 key/value——序列内部任意两个位置直接通信，距离不再是障碍。这就是 Transformer 的核心。

本节自下而上：scaled dot-product attention → 多头注意力 → Transformer Encoder block → 在 chap8 上的情感任务上对比 BiLSTM + attention vs 纯 Transformer。

## 1. Scaled Dot-Product Attention

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

$\sqrt{d_k}$ 缩放是为了避免维度高时点积变得过大、softmax 出现极端集中。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

def scaled_dot_attention(Q, K, V, mask=None):
    """Q,K,V: [..., L, d];  mask: [..., L_q, L_k] (True=valid)"""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)        # [..., L_q, L_k]
    if mask is not None:
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    attn = F.softmax(scores, dim=-1)
    return attn @ V, attn

# 验证：Q=K=V 时，每个位置应当注意到自己（如果其它位置很不像）
torch.manual_seed(0)
X = torch.eye(4) + 0.01 * torch.randn(4, 4)        # 近似 4 个正交单位向量
out, A = scaled_dot_attention(X, X, X)
print('attention matrix (diagonal should dominate):')
print(A.round(decimals=3))

## 2. 多头注意力

把 $Q, K, V$ 投影到 $h$ 个低维子空间，并行做 attention，再拼回来。让模型在不同子空间关注不同类型的关系（句法、语义、位置等）。

PyTorch 内置 `nn.MultiheadAttention`，但出于学习目的，我们手写一遍。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

    def _split_heads(self, x):                # [B, L, embed] -> [B, n_heads, L, head_dim]
        B, L, _ = x.shape
        return x.reshape(B, L, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, key_padding_mask=None):
        # key_padding_mask: [B, L]  True=valid
        Q = self._split_heads(self.W_q(x))
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))

        mask = None
        if key_padding_mask is not None:
            # 扩展到 [B, 1, 1, L]，broadcast 到所有 head 和 query 位置
            mask = key_padding_mask[:, None, None, :]

        out, attn = scaled_dot_attention(Q, K, V, mask=mask)   # out: [B, h, L, head_dim]
        B, h, L, d = out.shape
        out = out.transpose(1, 2).reshape(B, L, h * d)
        return self.W_o(out), attn                            # attn: [B, h, L, L]

mha = MultiHeadAttention(embed_dim=32, n_heads=4)
x = torch.randn(2, 7, 32)
out, attn = mha(x)
print('out:', tuple(out.shape), ' attn:', tuple(attn.shape))

## 3. 位置编码

自注意力本身**对位置无感**（permutation-equivariant）——打乱序列顺序，输出也只是相应地打乱。要让模型知道"第几个 token"，得额外加位置信号。

经典 sinusoidal positional encoding：
$$
PE_{pos, 2i} = \sin(pos / 10000^{2i/d}), \quad PE_{pos, 2i+1} = \cos(pos / 10000^{2i/d})
$$

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)        # 不参与训练

    def forward(self, x):                     # x: [B, L, d_model]
        return x + self.pe[:x.size(1)]

# 可视化前 100 个位置的前 16 维
pe = SinusoidalPE(d_model=64, max_len=100)
plt.figure(figsize=(10, 4))
plt.imshow(pe.pe[:100, :16].T, aspect='auto', cmap='RdBu_r')
plt.xlabel('position'); plt.ylabel('embedding dim'); plt.colorbar(); plt.title('sinusoidal positional encoding'); plt.show()

## 4. Transformer Encoder Block

标准结构：

```
x → MultiHeadAttn → +residual → LayerNorm
  → FeedForward    → +residual → LayerNorm
```

我们采用 **Pre-LN** 变体（LayerNorm 放在 sublayer 之前），训练更稳定，是 GPT-2 之后的现代默认。

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(),
            nn.Linear(ff_dim,  d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attn(self.norm1(x), key_padding_mask)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

## 5. 完整模型：Transformer 分类器

embedding → +PE → N 个 Encoder block → 取 padding-aware 的平均 → 线性分类。

In [ ]:
VOCAB, PAD = 25, 0
POS = [1, 2, 3]; NEG = [4, 5, 6]; FILL = list(range(7, VOCAB))

class TransformerClassifier(nn.Module):
    def __init__(self, vocab=VOCAB, d_model=64, n_heads=4, ff_dim=128, n_layers=2,
                 n_class=2, dropout=0.1, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.pe  = SinusoidalPE(d_model, max_len)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, ff_dim, dropout)
                                     for _ in range(n_layers)])
        self.norm  = nn.LayerNorm(d_model)
        self.fc    = nn.Linear(d_model, n_class)

    def forward(self, padded, lengths):
        mask = torch.arange(padded.size(1))[None, :] < lengths[:, None]   # [B, L] True=valid
        x = self.emb(padded)
        x = self.pe(x)
        for blk in self.blocks:
            x = blk(x, key_padding_mask=mask)
        x = self.norm(x)
        # padding-aware mean pooling
        m = mask.unsqueeze(-1).float()
        pooled = (x * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)
        return self.fc(pooled)

m = TransformerClassifier()
print('params:', sum(p.numel() for p in m.parameters()))

## 6. 训练（情感任务，与 chap8 上同一个数据生成器）

In [ ]:
def make_sample(g, L_range=(35, 46), n_sentiment=2):
    L = torch.randint(L_range[0], L_range[1], (1,), generator=g).item()
    label = torch.randint(0, 2, (1,), generator=g).item()
    tokens = [FILL[torch.randint(0, len(FILL), (1,), generator=g).item()] for _ in range(L)]
    sent_pool = POS if label == 1 else NEG
    positions = torch.randperm(L, generator=g)[:n_sentiment].tolist()
    for p in positions:
        tokens[p] = sent_pool[torch.randint(0, 3, (1,), generator=g).item()]
    return torch.tensor(tokens, dtype=torch.long), label

class SentimentDS(Dataset):
    def __init__(self, n, seed):
        g = torch.Generator().manual_seed(seed)
        self.data = [make_sample(g) for _ in range(n)]
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    return pad_sequence(seqs, batch_first=True, padding_value=PAD), lengths, torch.tensor(labels)

train_loader = DataLoader(SentimentDS(2000, seed=0), batch_size=64, shuffle=True, collate_fn=collate)
dev_loader   = DataLoader(SentimentDS(500,  seed=1), batch_size=64, collate_fn=collate)

torch.manual_seed(0)
model = TransformerClassifier()
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
accs = []
for ep in range(5):
    model.train()
    for x, l, y in train_loader:
        opt.zero_grad(); loss_fn(model(x, l), y).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, l, y in dev_loader:
            correct += (model(x, l).argmax(1) == y).sum().item(); total += y.size(0)
    accs.append(correct / total)
    print(f'epoch {ep+1}: dev acc = {accs[-1]:.4f}')

## 7. 看一下自注意力学到的关系

取一条 dev 样本，画第一层第一个 head 的注意力矩阵。如果模型学得好，**所有位置**的 query 都应该把高权重分配给 POS / NEG token 的位置。

In [ ]:
model.eval()
x_batch, len_batch, y_batch = next(iter(dev_loader))
x_one, len_one = x_batch[:1], len_batch[:1]

def token_kind(t):
    if t in POS: return '+'
    if t in NEG: return '-'
    return '.'

# 重新前向，并捕获第一个 block 的注意力
with torch.no_grad():
    x = model.emb(x_one); x = model.pe(x)
    mask = torch.arange(x_one.size(1))[None, :] < len_one[:, None]
    # 复刻 block forward, 拿 attention
    blk = model.blocks[0]
    h = blk.norm1(x)
    _, attn = blk.attn(h, key_padding_mask=mask)        # [B, n_heads, L, L]

L = len_one[0].item()
A = attn[0, 0, :L, :L]                                  # 第一个 head
tokens = x_one[0, :L].tolist()
kinds = [token_kind(t) for t in tokens]

plt.figure(figsize=(9, 8))
plt.imshow(A, cmap='hot', aspect='auto')
plt.colorbar(label='attention weight')
plt.xticks(range(L), [f'{k}{t}' for k, t in zip(kinds, tokens)], rotation=90, fontsize=7)
plt.yticks(range(L), [f'{k}{t}' for k, t in zip(kinds, tokens)], fontsize=7)
plt.xlabel('key (attended to)'); plt.ylabel('query (attending from)')
plt.title(f'block 0, head 0 attention  (true={y_batch[0].item()})')
plt.tight_layout(); plt.show()

亮列（key 维度）对应模型集中关注的位置；理想情况是 POS / NEG token 所在列普遍很亮。

## 8. 与 `nn.TransformerEncoder` 等价

PyTorch 自带 `nn.TransformerEncoderLayer` / `nn.TransformerEncoder`——实际项目直接用即可，比手写更优化（Flash Attention 等）。

In [ ]:
# 等价的 PyTorch 内置写法
layer = nn.TransformerEncoderLayer(
    d_model=64, nhead=4, dim_feedforward=128,
    dropout=0.1, batch_first=True, norm_first=True,        # Pre-LN
    activation='gelu',
)
encoder = nn.TransformerEncoder(layer, num_layers=2)

x = torch.randn(2, 10, 64)
pad_mask = torch.zeros(2, 10, dtype=torch.bool)
pad_mask[1, 7:] = True             # nn.Transformer: True 表示 *pad*（要忽略），与我们手写的相反
out = encoder(x, src_key_padding_mask=pad_mask)
print('nn.TransformerEncoder out:', tuple(out.shape))

**注意 mask 约定**：

- 我们手写：`mask[..., L] = True` 表示**有效**位置（attention 保留）
- `nn.TransformerEncoderLayer` 的 `src_key_padding_mask`：`True` 表示**应忽略**（padding 位置）

两种约定混用最容易翻车——切换 PyTorch 内置 API 时一定要记得 `~mask` 取反。

## 小结

- **Scaled dot-product attention**：$\text{softmax}(Q K^\top / \sqrt{d_k}) V$。$\sqrt{d_k}$ 缩放避免 softmax 集中。
- **多头**：把 query/key/value 切到 $h$ 个低维子空间并行做 attention，让模型在不同子空间关注不同类型的关系。
- **自注意力**：query=key=value=输入序列；序列内每两个位置直接通信，距离不再影响梯度。
- **位置编码**：自注意力本身位置无感，必须额外注入位置信号。Sinusoidal PE 不学习；learned PE 也常见；RoPE / ALiBi 是更现代的方案（chap8 PaddleEnding 之后）。
- **Transformer block**：MHA + FF + 两层 residual + LayerNorm。Pre-LN 比 Post-LN 训练更稳，是现代默认。
- **生产代码用 `nn.TransformerEncoderLayer`**——更快、更稳。但 mask 约定与教科书 / 大多数手写实现**相反**（True=ignore）。